# SILVER PA0008

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

source_file = "PA0008"
# Setup paths based on project structure
project_root = Path.cwd().parent
silver_dir = project_root / "data" / "02_silver" / "temporal_snapshots"

# Find the latest {source_file} parquet file
source_file_files = list(silver_dir.glob(f"*_{source_file}*.parquet"))

if not source_file_files:
    print(f"No {source_file} files found in silver layer!")
else:
    # Grab the most recent file if there are multiple
    file_path = str(source_file_files[-1])
    print(f"Targeting file: {file_path}")

In [ ]:
# Check total row count
print("Row Count:")
duckdb.sql(f"SELECT COUNT(*) as total_rows FROM '{file_path}'").show()

# Describe the schema
print("\nSchema:")
duckdb.sql(f"DESCRIBE SELECT * FROM '{file_path}'").show()

In [ ]:
# Fetch top 5 rows and display as a Pandas DataFrame for neat formatting
duckdb.sql(f"SELECT * FROM '{file_path}' LIMIT 5").df()

In [ ]:
# 1. First, get a list of all column names from the Parquet file
columns_df = duckdb.sql(f"DESCRIBE SELECT * FROM '{file_path}'").df()
column_names = columns_df['column_name'].tolist()

# 2. Build the dynamic SELECT list using Python list comprehension
# This loops through every column name and creates the SUM(CASE WHEN...) string for it
select_statements = [
    f"SUM(CASE WHEN TRIM(\"{col}\"::VARCHAR) = '' OR \"{col}\" IS NULL THEN 1 ELSE 0 END) AS \"{col}_blanks\""
    for col in column_names
]

# 3. Join them together with commas
dynamic_select_block = ",\n    ".join(select_statements)

# 4. Construct the final query
query = f"""
SELECT 
    COUNT(*) AS total_rows,
    {dynamic_select_block}
FROM '{file_path}'
"""

# Print the query so you can see what Python built for you (optional)
# print("Generated SQL Query:")
# print(query)

# 5. Execute the dynamically built query
df_profiling = duckdb.sql(query).df()

# Display the results transposed so it's easier to read (columns become rows)
display(df_profiling.T)